## Data, Baselines, and Experiments 1–3 (AAAC reference, OSACT benchmark, Unified benchmark)

data loading & harmonization, Category A/B/C model training, Experiments 1–3.

## 1. Setup & Install

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn pyarabic scipy statsmodels
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%matplotlib inline
import os, re, json, random, warnings, time, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import torch
from torch.utils.data import Dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
import joblib, pyarabic.araby as araby
from statsmodels.stats.contingency_tables import mcnemar

print("Imports OK.")


## 2. Config

In [ ]:
#  Paths 
DATASETS_BASE = "/content/drive/MyDrive/Datasets"

AAAC_DIR    = f"{DATASETS_BASE}/AAAC"
OSACT_DIR   = f"{DATASETS_BASE}/OSACT"
LAYER1_DIR  = f"{DATASETS_BASE}/layer1"

AAAC_TRAIN_PATH  = f"{AAAC_DIR}/train_aaac.csv"
AAAC_VAL_PATH    = f"{AAAC_DIR}/val_aaac.csv"
AAAC_TEST_PATH   = f"{AAAC_DIR}/test_aaac.csv"

OSACT_TRAIN_PATH = f"{OSACT_DIR}/train_osact.csv"
OSACT_DEV_PATH   = f"{OSACT_DIR}/dev_osact.csv"
OSACT_TEST_PATH  = f"{OSACT_DIR}/test_osact.csv"  

LAYER1_TRAIN_PATH = f"{LAYER1_DIR}/layer1_train.csv"
LAYER1_VAL_PATH   = f"{LAYER1_DIR}/layer1_validation.csv"
LAYER1_TEST_PATH  = f"{LAYER1_DIR}/layer1_test.csv"


PUBLISHED_AAAC_RESULTS_DIR = f"{DATASETS_BASE}/AAAC/outputs/results"
PUBLISHED_AAAC_MODELS_DIR  = f"{DATASETS_BASE}/AAAC/outputs/models/adversarial"   
PUBLISHED_AAAC_TFIDF_PATH  = f"{DATASETS_BASE}/AAAC/outputs/models/tfidf_vectorizer.joblib"  

#  Run config ─
SEED          = 42
N_SEEDS       = 3          
BOOTSTRAP_N   = 1000
MAX_LENGTH    = 64
BATCH_SIZE    = 16
EPOCHS        = 4
LEARNING_RATE = 2e-5
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

TRANSFORMER_MODELS = {
    "MARBERT":   "UBC-NLP/MARBERT",
    "AraBERT":   "aubmindlab/bert-base-arabertv02",
    "DziriBERT": "alger-ia/dziribert",
    "XLM-R":     "xlm-roberta-base",   # Category C — confirmed Base only, no Large
}
ARABIC_MODELS       = ["MARBERT", "AraBERT", "DziriBERT"]   # Category B
MULTILINGUAL_MODELS = ["XLM-R"]                              # Category C

#  Output dirs (all under Datasets/layer1/outputs so 05b/05c can find them) 
OUTPUT_DIR   = f"{LAYER1_DIR}/outputs"
MODELS_DIR   = f"{OUTPUT_DIR}/models"
FIGS_DIR     = f"{OUTPUT_DIR}/figures/05a"
RESULTS_DIR  = f"{OUTPUT_DIR}/results"
PREDS_DIR    = f"{OUTPUT_DIR}/predictions"     # raw y_true/y_pred/y_proba arrays for 05b/05c
for d in [OUTPUT_DIR, MODELS_DIR, FIGS_DIR, RESULTS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Config loaded. Device:", DEVICE)


## 3. Load Raw Data

In [ ]:
aaac_train = pd.read_csv(AAAC_TRAIN_PATH)
aaac_val   = pd.read_csv(AAAC_VAL_PATH)
aaac_test  = pd.read_csv(AAAC_TEST_PATH)
for _df in (aaac_train, aaac_val, aaac_test):
    _df["is_adversarial"] = (_df["category"] != "safe").astype(int)

osact_train = pd.read_csv(OSACT_TRAIN_PATH)
osact_dev   = pd.read_csv(OSACT_DEV_PATH)

layer1_train = pd.read_csv(LAYER1_TRAIN_PATH)
layer1_val   = pd.read_csv(LAYER1_VAL_PATH)
layer1_test  = pd.read_csv(LAYER1_TEST_PATH)

print("AAAC     :", aaac_train.shape, aaac_val.shape, aaac_test.shape)
print("OSACT    :", osact_train.shape, osact_dev.shape)
print("Unified  :", layer1_train.shape, layer1_val.shape, layer1_test.shape)

print("\n--- AAAC is_adversarial value counts (train) ---")
print(aaac_train["is_adversarial"].value_counts(dropna=False))

print("\n--- OSACT layer1_label value counts (train) ---")
print(osact_train["layer1_label"].value_counts(dropna=False))

print("\n--- Unified binary_label value counts (train) ---")
print(layer1_train["binary_label"].value_counts(dropna=False))

print("\n--- Unified language_variant value counts (train) ---")
print(layer1_train["language_variant"].value_counts(dropna=False))

print("\n--- Unified source value counts (train) ---")
print(layer1_train["source"].value_counts(dropna=False))


## 4. Label & Text Harmonization

In [ ]:
arabic_diacritics = re.compile(r"[\u064B-\u0652\u0640]")
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
EMOJI_RE   = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2700-\u27BF]+",
    flags=re.UNICODE
)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = re.sub(arabic_diacritics, "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text); text = re.sub(r"[ؤئ]", "ء", text); text = re.sub(r"ة", "ه", text)
    text = araby.strip_tatweel(text)
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    text = text.replace("_", " ")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

OSACT_LABEL_MAP = {
    "safe": 0,
    "hate_speech": 1,
}

UNIFIED_LABEL_MAP = {
    "safe": 0,
    "adversarial": 1,
}

def harmonize(df, text_col, label_col, label_map=None, variant_col=None,
              default_variant=None, source_name=""):
    out = pd.DataFrame()
    out["text_raw"] = df[text_col].astype(str)
    out["clean_text"] = out["text_raw"].apply(clean_text)
    if label_map is not None:
        out["label"] = df[label_col].map(label_map)
        n_unmapped = out["label"].isna().sum()
        if n_unmapped:
            print(f"[WARN] {source_name}: {n_unmapped} rows had a label value not in the map — "
                  f"unique unmapped values: {df.loc[out['label'].isna(), label_col].unique()[:10]}")
    else:
        out["label"] = df[label_col].astype(int)
    out["language_variant"] = df[variant_col] if variant_col is not None else default_variant
    out["source"] = source_name
    before = len(out)
    out = out.dropna(subset=["label"]).reset_index(drop=True)
    out["label"] = out["label"].astype(int)
    out = out[out["clean_text"].str.len() > 0].reset_index(drop=True)
    after = len(out)
    if after < before:
        print(f"  {source_name}: dropped {before - after} rows (unmapped label or empty clean_text)")
    return out


#  AAAC 
h_aaac_train = harmonize(aaac_train, "text", "is_adversarial", variant_col="language_variant", source_name="aaac")
h_aaac_val   = harmonize(aaac_val,   "text", "is_adversarial", variant_col="language_variant", source_name="aaac")
h_aaac_test  = harmonize(aaac_test,  "text", "is_adversarial", variant_col="language_variant", source_name="aaac")

#  OSACT 
h_osact_train = harmonize(osact_train, "text", "layer1_label", label_map=OSACT_LABEL_MAP,
                           default_variant="osact_msa", source_name="osact")
h_osact_dev   = harmonize(osact_dev,   "text", "layer1_label", label_map=OSACT_LABEL_MAP,
                           default_variant="osact_msa", source_name="osact")

#  Unified Layer1 (already harmonized upstream — binary_label + language_variant present) 
h_unified_train = harmonize(layer1_train, "text", "binary_label", label_map=UNIFIED_LABEL_MAP, variant_col="language_variant", source_name="unified")
h_unified_val   = harmonize(layer1_val,   "text", "binary_label", label_map=UNIFIED_LABEL_MAP, variant_col="language_variant", source_name="unified")
h_unified_test  = harmonize(layer1_test,  "text", "binary_label", label_map=UNIFIED_LABEL_MAP, variant_col="language_variant", source_name="unified")

print("\nHarmonized shapes:")
print("  AAAC   train/val/test :", h_aaac_train.shape, h_aaac_val.shape, h_aaac_test.shape)
print("  OSACT  train/dev      :", h_osact_train.shape, h_osact_dev.shape)
print("  Unified train/val/test:", h_unified_train.shape, h_unified_val.shape, h_unified_test.shape)


## 5. Statistical Utilities (bootstrap CI, McNemar, per-class table)

In [ ]:
def bootstrap_macro_f1_ci(y_true, y_pred, n_bootstrap=BOOTSTRAP_N, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        yt, yp = np.array(y_true)[idx], np.array(y_pred)[idx]
        _, _, f1, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
        scores.append(f1)
    mean_f1 = np.mean(scores)
    ci_low  = np.percentile(scores, 100 * alpha / 2)
    ci_high = np.percentile(scores, 100 * (1 - alpha / 2))
    return mean_f1, ci_low, ci_high


def run_mcnemar(y_true, y_pred_a, y_pred_b, model_a="A", model_b="B", tag=""):
    y_true = np.array(y_true)
    correct_a = (np.array(y_pred_a) == y_true)
    correct_b = (np.array(y_pred_b) == y_true)
    b = np.sum(correct_a & ~correct_b)
    c = np.sum(~correct_a & correct_b)
    table = [[np.sum(correct_a & correct_b), b], [c, np.sum(~correct_a & ~correct_b)]]
    result = mcnemar(table, exact=(b + c < 25))
    sig = "significant" if result.pvalue < 0.05 else "not significant"
    print(f"[{tag}] McNemar {model_a} vs {model_b}: b={b}, c={c}, p={result.pvalue:.4f} -> {sig} (a=0.05)")
    return result.pvalue


LABEL_NAMES = ["benign", "adversarial"]


def per_class_table(y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=LABEL_NAMES,
                                    output_dict=True, zero_division=0)
    rows = []
    for cls in LABEL_NAMES:
        row = report[cls]
        row["class"] = cls
        rows.append(row)
    return pd.DataFrame(rows).set_index("class")[["precision", "recall", "f1-score", "support"]]


def compute_all_metrics(y_true, y_pred, y_proba=None, model_name="", tag=""):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    _, ci_lo, ci_hi = bootstrap_macro_f1_ci(y_true, y_pred)
    roc_auc, pr_auc = None, None
    if y_proba is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_proba)
            pr_auc  = average_precision_score(y_true, y_proba)
        except Exception as e:
            print(f"  [WARN] could not compute ROC/PR AUC for {model_name}: {e}")
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"\n===== [{tag}] {model_name} =====")
    print(f"Macro F1: {f1:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]" +
          (f"  ROC AUC: {roc_auc:.4f}  PR AUC: {pr_auc:.4f}" if roc_auc is not None else ""))
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))
    return {
        "accuracy": acc, "precision": precision, "recall": recall,
        "macro_f1": f1, "weighted_f1": f1_w,
        "f1_ci_low": ci_lo, "f1_ci_high": ci_hi,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "balanced_accuracy": balanced_acc, "mcc": mcc, "cohen_kappa": kappa,
        "confusion_matrix": cm.tolist(),
        "y_true": [int(v) for v in y_true],
        "y_pred": [int(v) for v in y_pred],
        "y_proba": [float(v) for v in y_proba] if y_proba is not None else None,
    }


print("Statistical utilities defined.")


## 6. Results Store & Persistence Helpers

In [ ]:
results = {}

def save_experiment(exp_key):
    '''Persist one experiment's results dict to Drive (JSON + pickle) so 05b/05c can reload it.'''
    path_json = f"{RESULTS_DIR}/{exp_key}_results.json"
    path_pkl  = f"{PREDS_DIR}/{exp_key}_results.pkl"
    with open(path_json, "w") as f:
        json.dump(results[exp_key], f)
    with open(path_pkl, "wb") as f:
        pickle.dump(results[exp_key], f)
    print(f"Saved experiment '{exp_key}' -> {path_json}")


def save_partial(exp_key, model_key):
    '''Save results[exp_key] to Drive immediately after a single model finishes, so a Colab
    disconnect mid-experiment only loses the model currently in flight, not the whole experiment.
    Safe to call repeatedly — overwrites the same {exp_key}_results.json/.pkl each time.
    '''
    save_experiment(exp_key)
    print(f"  [checkpoint] '{exp_key}' saved after '{model_key}' ({len(results.get(exp_key, {}))} models so far)")


def load_experiment(exp_key):
    path_pkl = f"{PREDS_DIR}/{exp_key}_results.pkl"
    with open(path_pkl, "rb") as f:
        return pickle.load(f)


def resume_experiment(exp_key):
    '''Call at the top of an experiment cell to reload any partial results already saved to Drive
    (e.g. after a disconnect), so already-finished models are not retrained.
    '''
    path_pkl = f"{PREDS_DIR}/{exp_key}_results.pkl"
    if os.path.exists(path_pkl):
        results[exp_key] = load_experiment(exp_key)
        print(f"Resumed '{exp_key}' with {len(results[exp_key])} model(s) already done: "
              f"{list(results[exp_key].keys())}")
    else:
        results.setdefault(exp_key, {})
        print(f"No prior partial results found for '{exp_key}' — starting fresh.")


print("Persistence helpers defined.")


## 7. PyTorch Dataset

In [8]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(list(texts), truncation=True,
                                    padding="max_length", max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


## 8. Category A — Classical ML Baselines (LogReg + Linear SVM)

In [9]:
def train_ml_baselines(train_df, test_df, exp_key, val_df=None, only_models=None):
    '''Fits TF-IDF (word 1-2 grams, max_features=50000, lowercase=False per spec) +
    Logistic Regression and Linear SVM. Reports full metric set incl. ROC/PR AUC via decision_function.
    `only_models`: optional list to restrict which classical models are trained (e.g. ["LogReg"]).
    '''
    tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), lowercase=False)
    X_train = tfidf.fit_transform(train_df["clean_text"])
    X_test  = tfidf.transform(test_df["clean_text"])
    y_train = train_df["label"].values
    y_test  = test_df["label"].values

    os.makedirs(f"{MODELS_DIR}/{exp_key}", exist_ok=True)
    joblib.dump(tfidf, f"{MODELS_DIR}/{exp_key}/tfidf_vectorizer.joblib")

    configs = {
        "LogReg": LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=SEED),
        "SVM":    LinearSVC(C=1.0, class_weight="balanced", random_state=SEED),
    }
    if only_models is not None:
        configs = {k: v for k, v in configs.items() if k in only_models}
    for model_key, clf in configs.items():
        if model_key in results.get(exp_key, {}):
            print(f"[skip] '{exp_key}'/'{model_key}' already completed (found in results) — not retraining.")
            continue
        print(f"\n{'='*70}\n[{exp_key}] {model_key}\n{'='*70}")
        t0 = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - t0

        t0 = time.time()
        y_pred = clf.predict(X_test)
        infer_time = time.time() - t0

        if hasattr(clf, "predict_proba"):
            y_proba = clf.predict_proba(X_test)[:, 1]
        else:
            scores = clf.decision_function(X_test)
            y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

        clf_path = f"{MODELS_DIR}/{exp_key}/{model_key}.joblib"
        joblib.dump(clf, clf_path)
        model_mb = os.path.getsize(clf_path) / 1e6

        m = compute_all_metrics(y_test, y_pred, y_proba, model_key, exp_key)
        m["train_time_s"]  = round(train_time, 4)
        m["infer_time_s"]  = round(infer_time, 4)
        m["model_mb"]      = round(model_mb, 3)
        m["n_params"]      = int(X_train.shape[1])  # feature count as proxy; classical models have no "parameters"
        results.setdefault(exp_key, {})[model_key] = m
        save_partial(exp_key, model_key)
    return tfidf


## 9. Category B/C — Transformer Fine-Tuning (Arabic + Multilingual, multi-seed)

In [ ]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (self.class_weights_tensor.to(logits.device)
                   if self.class_weights_tensor is not None else None)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(
            logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def make_compute_metrics_fn():
    def _compute(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        acc = accuracy_score(labels, preds)
        _, _, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
        return {"accuracy": acc, "macro_f1": f1}
    return _compute


def train_transformer_model(model_key, checkpoint, train_df, val_df, test_df, exp_key, seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*70}\n[{exp_key}] {model_key}  seed={seed}\n{'='*70}")

    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(DEVICE)

    train_ds = TextClassificationDataset(train_df["clean_text"], train_df["label"], tokenizer)
    val_ds   = TextClassificationDataset(val_df["clean_text"],   val_df["label"],   tokenizer)
    test_ds  = TextClassificationDataset(test_df["clean_text"],  test_df["label"],  tokenizer)

    model_out_dir = f"{MODELS_DIR}/{exp_key}/{model_key}"
    os.makedirs(model_out_dir, exist_ok=True)

    classes = np.unique(train_df["label"].values)
    cw = compute_class_weight("balanced", classes=classes, y=train_df["label"].values)
    cw_tensor = torch.tensor(cw, dtype=torch.float)

    args = TrainingArguments(
        output_dir=f"/content/checkpoints/{exp_key}_{model_key}_s{seed}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        eval_strategy="epoch",
        save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = WeightedLossTrainer(
        model=model, args=args, class_weights_tensor=cw_tensor,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=make_compute_metrics_fn(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    t_infer0 = time.time()
    preds_out = trainer.predict(test_ds)
    infer_time = time.time() - t_infer0
    logits = preds_out.predictions
    y_pred = np.argmax(logits, axis=1)
    y_proba = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    y_true = test_df["label"].values

    elapsed = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    model_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
    gpu_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if torch.cuda.is_available() else None
    print(f"  Training time: {elapsed:.0f}s | Inference time: {infer_time:.2f}s | "
          f"Model size: {model_mb:.1f} MB | Params: {n_params/1e6:.1f}M")

    # IMPORTANT: metrics are computed for EVERY seed (not just the canonical one) so that
    # multi-seed mean/std aggregation in train_transformer_multiseed is real, not a single-value
    # std=0 artifact. Only the canonical seed's checkpoint is written to disk (avoids 3x storage
    # cost per model); only the canonical seed's raw y_true/y_pred/y_proba/confusion_matrix are
    # kept (needed for downstream reuse in 05b) — non-canonical seeds keep scalar metrics only.
    m = compute_all_metrics(y_true, y_pred, y_proba, f"{model_key} (seed={seed})", exp_key)
    m["train_time_s"] = round(elapsed, 1)
    m["infer_time_s"] = round(infer_time, 3)
    m["model_mb"]     = round(model_mb, 1)
    m["n_params"]     = int(n_params)
    m["gpu_mem_mb"]   = round(gpu_mem_mb, 1) if gpu_mem_mb is not None else None
    m["seed"]         = seed

    if seed == SEED:
        trainer.save_model(model_out_dir)
    else:
        m = {k: v for k, v in m.items() if k not in ("y_true", "y_pred", "y_proba", "confusion_matrix")}
    out = m

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    import shutil
    shutil.rmtree(args.output_dir, ignore_errors=True)
    return y_pred, y_true, y_proba, out


AGGREGATABLE_METRICS = [
    "accuracy", "precision", "recall", "macro_f1", "weighted_f1",
    "roc_auc", "pr_auc", "balanced_accuracy", "mcc", "cohen_kappa",
    "train_time_s", "infer_time_s",
]


def train_transformer_multiseed(model_key, checkpoint, train_df, val_df, test_df, exp_key):
    existing = results.get(exp_key, {}).get(model_key)
    if existing is not None and "per_seed" in existing and len(existing["per_seed"]) >= N_SEEDS:
        print(f"[skip] '{exp_key}'/'{model_key}' already has a complete {N_SEEDS}-seed aggregation — not retraining.")
        return

    per_seed = {}
    canonical_metrics = None
    for seed in range(SEED, SEED + N_SEEDS):
        _, y_true, _, m = train_transformer_model(model_key, checkpoint, train_df, val_df, test_df, exp_key, seed=seed)
        per_seed[seed] = m
        if seed == SEED:
            canonical_metrics = dict(m)  # keeps arrays (y_true/y_pred/y_proba/confusion_matrix)

    if canonical_metrics is None:
        print(f"  -> [{exp_key}] {model_key}: no canonical result produced, skipping aggregation.")
        return

    for metric in AGGREGATABLE_METRICS:
        values = [per_seed[s].get(metric) for s in per_seed if per_seed[s].get(metric) is not None]
        if not values:
            continue
        mean_v, std_v = float(np.mean(values)), float(np.std(values))
        canonical_metrics[f"{metric}_mean"] = round(mean_v, 4)
        canonical_metrics[f"{metric}_std"]  = round(std_v, 4)
        if len(values) > 1 and std_v == 0.0 and len(set(round(v, 6) for v in values)) > 1:
            print(f"  [WARN] {exp_key}/{model_key}: std==0 for '{metric}' but per-seed values differ "
                  f"({values}) — this should not happen, please investigate.")

    canonical_metrics["per_seed"] = per_seed   # full per-seed record — never overwritten

    results.setdefault(exp_key, {})[model_key] = canonical_metrics
    save_partial(exp_key, model_key)
    seed_f1s = [per_seed[s]["macro_f1"] for s in per_seed]
    print(f"  -> [{exp_key}] {model_key} Macro F1 per seed: "
          f"{ {s: round(per_seed[s]['macro_f1'], 4) for s in per_seed} } "
          f"| mean={np.mean(seed_f1s):.4f} std={np.std(seed_f1s):.4f}")


def run_all_transformers(train_df, val_df, test_df, exp_key, models=None):
    models = models or TRANSFORMER_MODELS
    for model_key, checkpoint in models.items():
        try:
            train_transformer_multiseed(model_key, checkpoint, train_df, val_df, test_df, exp_key)
        except Exception as e:
            print(f"[ERROR] {exp_key}/{model_key}: {e}")


print("Transformer training functions defined.")


## 10. Experiment 1 — AAAC Baseline (Reference Only, NOT Retrained)

In [ ]:
aaac_reference = None
try:
    with open(f"{PUBLISHED_AAAC_RESULTS_DIR}/adversarial_results.json") as f:
        aaac_reference = json.load(f)
    print("Loaded published AAAC (binary adversarial) reference results for:",
          list(aaac_reference.keys()))
except FileNotFoundError as e:
    print("[WARN] Could not find published AAAC results at the configured path:")
    print(" ", e)
    print("  Update PUBLISHED_AAAC_RESULTS_DIR in the Config cell to point at your "
          "04_main_experiments.ipynb RESULTS_DIR, then re-run this cell.")
    print("  Experiment 1 will be skipped in downstream comparison tables until this is fixed.")

if aaac_reference is not None:
    results["aaac_reference"] = aaac_reference
    save_experiment("aaac_reference")


## 11. Experiment 2 — OSACT Benchmark

In [ ]:
EXP2_KEY = "osact"
resume_experiment(EXP2_KEY)  # reloads any models already finished from a prior/interrupted run

osact_train_sub, osact_val_sub = train_test_split(
    h_osact_train, test_size=0.10, stratify=h_osact_train["label"], random_state=SEED
)
print(f"OSACT train_sub: {osact_train_sub.shape} | OSACT val_sub (early-stop only): {osact_val_sub.shape} "
      f"| OSACT dev (final scoring): {h_osact_dev.shape}")

print("\n### Category A: Classical baselines (OSACT) ###")
train_ml_baselines(osact_train_sub, h_osact_dev, EXP2_KEY, val_df=osact_val_sub)

print("\n### Category B/C: Transformer fine-tuning (OSACT) ###")
run_all_transformers(osact_train_sub, osact_val_sub, h_osact_dev, EXP2_KEY)

save_experiment(EXP2_KEY)


## 12. Experiment 3 — Unified Dataset Benchmark

In [ ]:
EXP3_KEY = "unified"
resume_experiment(EXP3_KEY)  # reloads any models already finished from a prior/interrupted run

print("### Category A: Classical baselines (Unified) ###")
train_ml_baselines(h_unified_train, h_unified_test, EXP3_KEY, val_df=h_unified_val)

print("\n### Category B/C: Transformer fine-tuning (Unified) ###")
run_all_transformers(h_unified_train, h_unified_val, h_unified_test, EXP3_KEY)

save_experiment(EXP3_KEY)


## 13. Experiments 1–3 Summary Table (Category D preview)

In [ ]:
def fmt_mean_std(m, metric, force_single=False):
    mean_v, std_v = m.get(f"{metric}_mean"), m.get(f"{metric}_std")
    if mean_v is not None and std_v is not None and not force_single:
        return f"{mean_v:.4f} +- {std_v:.4f}"
    val = mean_v if mean_v is not None else m.get(metric)
    return f"{val:.4f}" if isinstance(val, (int, float)) else "-"


def build_summary_table(exp_key):
    # aaac_reference is loaded verbatim from the PUBLISHED benchmark's results.json, which stores
    # macro_f1_mean/_std but no per_seed list. That means `is_multiseed` (which gates the "N seeds"
    # column) reads False -> "N seeds"=1, while fmt_mean_std still finds *_mean/*_std and prints a
    # "+-" spread -- a 1-seed row with a 3-seed-looking CI is misleading in a reproducibility
    # appendix. Fix: for this experiment only, force every metric to print as a single value (no
    # +-), and print an explicit footnote pointing to where the real multi-seed spread lives.
    is_reference_exp = (exp_key == "aaac_reference")

    rows = []
    for model_name, m in results.get(exp_key, {}).items():
        ci_lo = m.get("f1_ci_low", float("nan"))
        ci_hi = m.get("f1_ci_high", float("nan"))
        is_multiseed = "per_seed" in m and len(m["per_seed"]) > 1
        rows.append({
            "Experiment": exp_key,
            "Model": model_name,
            "N seeds": len(m.get("per_seed", {})) if is_multiseed else 1,
            "Accuracy": fmt_mean_std(m, "accuracy", force_single=is_reference_exp),
            "Precision": fmt_mean_std(m, "precision", force_single=is_reference_exp),
            "Recall": fmt_mean_std(m, "recall", force_single=is_reference_exp),
            "Macro F1": fmt_mean_std(m, "macro_f1", force_single=is_reference_exp),
            "Weighted F1": fmt_mean_std(m, "weighted_f1", force_single=is_reference_exp),
            "ROC AUC": fmt_mean_std(m, "roc_auc", force_single=is_reference_exp),
            "PR AUC": fmt_mean_std(m, "pr_auc", force_single=is_reference_exp),
            "Balanced Acc.": fmt_mean_std(m, "balanced_accuracy", force_single=is_reference_exp),
            "MCC": fmt_mean_std(m, "mcc", force_single=is_reference_exp),
            "Cohen's Kappa": fmt_mean_std(m, "cohen_kappa", force_single=is_reference_exp),
            "Canonical-seed F1 95% CI": f"[{ci_lo:.4f}, {ci_hi:.4f}]",
            "Train time (s)": fmt_mean_std(m, "train_time_s", force_single=is_reference_exp) if is_multiseed else m.get("train_time_s", "-"),
            "Infer time (s)": m.get("infer_time_s", "-"),
            "Model MB": m.get("model_mb", "-"),
            "Params": m.get("n_params", "-"),
        })
    df = pd.DataFrame(rows)
    sort_key = df["Macro F1"].apply(lambda s: float(s.split(" +- ")[0]) if isinstance(s, str) and s != "-" else (float(s) if isinstance(s, str) and s != "-" else -1))
    df = df.iloc[sort_key.sort_values(ascending=False).index].reset_index(drop=True)
    df.to_csv(f"{RESULTS_DIR}/{exp_key}_summary.csv", index=False)
    return df, is_reference_exp


for exp_key in ["aaac_reference", "osact", "unified", "aaac_transfer"]:
    if exp_key in results:
        print(f"\n=== {exp_key} ===")
        summary_df, is_ref = build_summary_table(exp_key)
        display(summary_df)
        if is_ref:
            print("Note: aaac_reference rows are loaded verbatim from the PUBLISHED AAAC benchmark "
                  "(04_main_experiments.ipynb) and are NOT retrained here, so no per-seed spread is "
                  "available in this notebook -- values above are single canonical-run numbers, not "
                  "mean +- std. The published run's own multi-seed std (if you need it for the paper) "
                  "lives in 04_main_experiments.ipynb's saved results, not here.")


## 14. Additional AAAC Training for Cross-Dataset Transfer

**Scope note**: this section trains **only** the two models that were never part of the published AAAC
benchmark — Logistic Regression + TF-IDF, and XLM-R Base. SVM, MARBERT, AraBERT, and DziriBERT are **not**
retrained here; their published checkpoints (`PUBLISHED_AAAC_MODELS_DIR`) are reused as-is.

These two new "transfer" models are saved under a clearly separate experiment key (`aaac_transfer`, i.e.
`outputs/models/aaac_transfer/`), so they can never be confused with the original published AAAC benchmark.
Their only purpose is to let `05b`'s Experiment 4 (cross-dataset generalization) and Experiment 9 (ablation)
evaluate all six models fairly across AAAC / OSACT / Unified, instead of leaving LogReg/XLM-R with an
AAAC-shaped hole in the comparison.


In [ ]:
EXP_TRANSFER_KEY = "aaac_transfer"
resume_experiment(EXP_TRANSFER_KEY)

print("### LogReg + TF-IDF on AAAC (transfer model only — SVM is NOT retrained here) ###")
train_ml_baselines(h_aaac_train, h_aaac_test, EXP_TRANSFER_KEY, val_df=h_aaac_val, only_models=["LogReg"])

print("\n### XLM-R Base on AAAC (transfer model only — MARBERT/AraBERT/DziriBERT are NOT retrained here) ###")
run_all_transformers(h_aaac_train, h_aaac_val, h_aaac_test, EXP_TRANSFER_KEY,
                      models={"XLM-R": TRANSFORMER_MODELS["XLM-R"]})

save_experiment(EXP_TRANSFER_KEY)
if EXP_TRANSFER_KEY in results:
    summary_df, _ = build_summary_table(EXP_TRANSFER_KEY)
    display(summary_df)